# Prophet hyperparameter search

Reworks the original 72-combination search into a focused validation grid. Candidates are scored only on the middle 15% of the time series. The best configuration is then refitted on the first 85%, and the final 15% is evaluated once.

The reconstructed CSV is included, but this notebook has no saved model results yet.


In [ ]:
from prophet import Prophet
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_absolute_error


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [Path("../data/clean_hourly_data.csv"), Path("data/clean_hourly_data.csv")]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("The reconstructed CSV is missing; see data/README.md")

data = pd.read_csv(data_path, parse_dates=["ds"])
required = {"ds", "y"}
if not required.issubset(data.columns):
    raise ValueError("The CSV must contain ds (timestamp) and y (numeric target) columns")
data = data[["ds", "y"]].copy().sort_values("ds").reset_index(drop=True)
data["y"] = pd.to_numeric(data["y"], errors="raise")
if data.isna().any().any() or data["ds"].duplicated().any():
    raise ValueError("Remove missing values and duplicate timestamps before running these notebooks")
if not data["ds"].diff().dropna().eq(pd.Timedelta(hours=1)).all():
    raise ValueError("These notebooks require consecutive hourly observations")

train_end = int(len(data) * 0.70)
validation_end = int(len(data) * 0.85)
if train_end < 48 or validation_end - train_end < 24 or len(data) - validation_end < 24:
    raise ValueError("The dataset is too short for the chronological split")
print(f"Rows: {len(data):,}; train: {train_end:,}; validation: {validation_end - train_end:,}; test: {len(data) - validation_end:,}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

def report(y_true, y_pred, timestamps, title):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    print(f"{title} | MAE: {mae:.3f} | RMSE: {rmse:.3f}")
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(timestamps, y_true, label="Observed", linewidth=1)
    ax.plot(timestamps, y_pred, label="Predicted", linewidth=1)
    ax.set(title=title, xlabel="Time", ylabel="Hourly sum of minute-level kW")
    ax.legend()
    fig.tight_layout()
    plt.show()


## Select parameters on validation data


In [ ]:
train = data.iloc[:train_end].copy()
validation = data.iloc[train_end:validation_end].copy()
test = data.iloc[validation_end:].copy()

grid = ParameterGrid({
    "changepoint_prior_scale": [0.01, 0.05, 0.5],
    "seasonality_prior_scale": [0.1, 1.0, 10.0],
})
scores = []
for params in grid:
    candidate = Prophet(**params, daily_seasonality=True, weekly_seasonality=True)
    candidate.fit(train)
    validation_prediction = candidate.predict(validation[["ds"]])["yhat"]
    scores.append({**params, "validation_mae": mean_absolute_error(validation["y"], validation_prediction)})

ranking = pd.DataFrame(scores).sort_values("validation_mae").reset_index(drop=True)
display(ranking)
best = ranking.iloc[0]
best_params = {
    "changepoint_prior_scale": float(best["changepoint_prior_scale"]),
    "seasonality_prior_scale": float(best["seasonality_prior_scale"]),
}
print("Selected parameters:", best_params)


## Refit and evaluate once on test


In [ ]:
final_model = Prophet(**best_params, daily_seasonality=True, weekly_seasonality=True)
final_model.fit(data.iloc[:validation_end].copy())
test_forecast = final_model.predict(test[["ds"]])
report(test["y"].to_numpy(), test_forecast["yhat"].to_numpy(),
       test["ds"], "Tuned Prophet: direct test-horizon forecast")


**Scope:** The original US-holiday feature was removed because the source household is in France. This experiment tunes only trend and seasonality; the test period has not influenced parameter selection.
